# Aplicação completa: API, banco de dados e modelo

Agora vamos unir os conceitos em uma aplicação de manutenção preditiva.


```bash
Cliente
   │
   ▼
API-FastAPI
   │
   ▼
Serviço-de-predição
   ├── Modelo-de-Machine-Learning
   └── Repositório-SQLite
```

Cada componente terá uma responsabilidade diferente.

In [2]:
# 1. Entidade de dominio
# A entidade representa dados recebidos dos sensores.

class LeituraSensor:
    def __init__(
        self,
        equipamento_id: str,
        temperatura: float,
        vibracao: float,
        horas_uso: float  
    ):

        self.equipameto_id = equipamento_id
        self.temperatura = float(temperatura)
        self.vibracao = float(vibracao)
        self.horas_uso = float(horas_uso)

        self._validar()

        def _valdiar(self) -> None:
            if not self.equipamento_id.strip():
                raise ValueError(
                    "O identificador é obrigatório."
                )

            if self.vibracao < 0:
                raise ValueError(
                    "A vibração não pode ser negativa."
                )

            if self.horas_uso < 0:
                raise ValueError(
                    "As horas de uso não podem ser negativas."
                )

        def transformar_em_vetor(self) -> list[list[float]]:
            return [[
                self.temperatura,
                self.vibracao,
                self.horas_uso
            ]]

"""
Essa classe:

    representa uma leitura;
    armazena os atributos;
    valida os dados;
    transforma os valores no formato esperado pelo modelo.
"""

'\nEssa classe:\n\n    representa uma leitura;\n    armazena os atributos;\n    valida os dados;\n    transforma os valores no formato esperado pelo modelo.\n'

In [3]:
# 2. Classe base para modelos

from abc import ABC, abstractclassmethod

class ModeloBase(ABC):
    @abstractclassmethod
    def prever_probabilidade(
        self,
        leitura: LeituraSensor
    ) -> float:
        pass

"""
Essa classe estabelece um contrato:

Todo modelo utilizado pelo sistema precisa implementar prever_probabilidade().

Ela não contém o algoritmo real.
"""

'\nEssa classe estabelece um contrato:\n\nTodo modelo utilizado pelo sistema precisa implementar prever_probabilidade().\n\nEla não contém o algoritmo real.\n'

In [4]:
# 3. Modelo que sobrescreve o método

class ModeloFalhaHeuristico(ModeloBase):
    def prever_probabilidade(
            self,
            leitura: LeituraSensor
    ) -> float:
        probabilidade = (
            leitura.temperatura / 100 * 0.40
            + leitura.vibracao / 10 * 0.35
            + leitura.horas_uso / 10_000 * 0.25
        )

        return min(
            max(probabilidade, 0.0),
            1.0
        )

"""
Aqui temos:

herança única;
sobrescrita de método;
polimorfismo.

O serviço poderá trabalhar com qualquer classe que respeite o contrato de ModeloBase.
"""

'\nAqui temos:\n\nherança única;\nsobrescrita de método;\npolimorfismo.\n\nO serviço poderá trabalhar com qualquer classe que respeite o contrato de ModeloBase.\n'

In [ ]:
## Usando um modelo real do scikit-learn

### Uma implementação real poderia receber um modelo já treinado.

class ModeloFalhaSklearn(ModeloBase):
    def __init__(self, modelo_treinado):
        self.modelo_treinado = modelo_treinado

    def prever_probabilidade(
            self,
            leitura: LeituraSensor
    ) -> float:
        vetor = leitura.transformar_em_vetor()

        probabilidades = (
            self.modelo_treinado.predict_rpoba(vetor)
        )

        return float(probabilidades[0][1])

"""
O objeto modelo_treinado poderia ser:

LogisticRegression;
RandomForestClassifier;
XGBClassifier;
um pipeline do scikit-learn;
um modelo carregado do MLflow.

O restante da aplicação não precisaria saber qual algoritmo está sendo utilizado.
"""

In [6]:
# 4. Classe base do repositório

from abc import ABC, abstractclassmethod

class RepositorioBase(ABC):
    def salvar(
            self,
            leitura: LeituraSensor,
            probabilidade: float
    ) -> int:
        pass

# Ela define o comportamento esperado para persistência.

In [ ]:
# 5. Repositório SQLite


import sqlite3


class RepositorioSQLite(RepositorioBase):
    def __init__(
        self,
        caminho_banco: str = "predicoes.db"
    ):
        self.caminho_banco = caminho_banco
        self._criar_tabela()

    def _conectar(self) -> sqlite3.Connection:
        return sqlite3.connect(
            self.caminho_banco
        )

    def _criar_tabela(self) -> None:
        sql = """
        CREATE TABLE IF NOT EXISTS predicoes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            equipamento_id TEXT NOT NULL,
            temperatura REAL NOT NULL,
            vibracao REAL NOT NULL,
            horas_uso REAL NOT NULL,
            probabilidade REAL NOT NULL
        )
        """

        with self._conectar() as conexao:
            conexao.execute(sql)

    def salvar(
        self,
        leitura: LeituraSensor,
        probabilidade: float
    ) -> int:
        sql = """
        INSERT INTO predicoes (
            equipamento_id,
            temperatura,
            vibracao,
            horas_uso,
            probabilidade
        )
        VALUES (?, ?, ?, ?, ?)
        """

        valores = (
            leitura.equipamento_id,
            leitura.temperatura,
            leitura.vibracao,
            leitura.horas_uso,
            probabilidade
        )

        with self._conectar() as conexao:
            cursor = conexao.execute(
                sql,
                valores
            )

            return int(cursor.lastrowid)


"""
Responsabilidade do repositório

O repositório é responsável por:

abrir a conexão;
criar a tabela;
executar SQL;
salvar os resultados;
retornar o identificador do registro.

O modelo não deve executar SQL.

A entidade não deve abrir conexões.

A API não deve conter todo o código de persistência.
"""

In [9]:
# 6. Mixins para validação e logs

from datetime import datetime


class LogMixin:
    def registrar_log(
        self,
        mensagem: str
    ) -> None:
        horario = datetime.now().isoformat(
            timespec="seconds"
        )

        print(f"[{horario}] {mensagem}")


class ValidacaoMixin:
    def validar_probabilidade(
        self,
        probabilidade: float
    ) -> None:
        if not 0 <= probabilidade <= 1:
            raise ValueError(
                "A probabilidade deve estar entre 0 e 1."
            )

In [10]:
# 7. Serviço usando herança múltipla e composição

class ServicoPredicao(
    LogMixin,
    ValidacaoMixin
):
    def __init__(
        self,
        modelo: ModeloBase,
        repositorio: RepositorioBase
    ):
        self.modelo = modelo
        self.repositorio = repositorio

    def executar(
        self,
        leitura: LeituraSensor
    ) -> dict:
        self.registrar_log(
            f"Analisando {leitura.equipamento_id}."
        )

        probabilidade = (
            self.modelo.prever_probabilidade(
                leitura
            )
        )

        self.validar_probabilidade(
            probabilidade
        )

        registro_id = self.repositorio.salvar(
            leitura,
            probabilidade
        )

        falha_prevista = probabilidade >= 0.5

        self.registrar_log(
            f"Predição salva com ID {registro_id}."
        )

        return {
            "registro_id": registro_id,
            "equipamento_id": leitura.equipamento_id,
            "probabilidade_falha": round(
                probabilidade,
                4
            ),
            "falha_prevista": falha_prevista
        }

In [11]:
# 8. Configuração privada da API

class ConfiguracaoAPI:
    __limite_requisicoes = 100

    @classmethod
    def obter_limite(cls) -> int:
        return cls.__limite_requisicoes

    @classmethod
    def alterar_limite(
        cls,
        novo_limite: int
    ) -> None:
        if novo_limite <= 0:
            raise ValueError(
                "O limite precisa ser positivo."
            )

        cls.__limite_requisicoes = novo_limite

In [12]:
!pip install fastapi 
!pip install pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 551.8 kB/s eta 0:00:00a 0:00:01
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 1.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 2.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 5.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 509.2 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 976.2 kB/s eta 0:00:000:01:00:01
Using cached idna-3.18-py3-none-any.whl (65 kB)


In [13]:
# 9. Criando a API com FastAPI

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


app = FastAPI(
    title="API de manutenção preditiva",
    version="1.0.0"
)

class LeituraEntrada(BaseModel):
    equipamento_id: str = Field(
        min_length=1
    )

    temperatura: float = Field(
        ge=-50,
        le=300
    )

    vibracao: float = Field(
        ge=0
    )

    horas_uso: float = Field(
        ge=0
    )

In [14]:
modelo = ModeloFalhaHeuristico()

repositorio = RepositorioSQLite(
    "predicoes.db"
)

servico = ServicoPredicao(
    modelo=modelo,
    repositorio=repositorio
)

In [15]:
@app.post("/predicoes")
def criar_predicao(
    dados: LeituraEntrada
):
    try:
        leitura = LeituraSensor(
            equipamento_id=dados.equipamento_id,
            temperatura=dados.temperatura,
            vibracao=dados.vibracao,
            horas_uso=dados.horas_uso
        )

        return servico.executar(leitura)

    except ValueError as erro:
        raise HTTPException(
            status_code=400,
            detail=str(erro)
        ) from erro

## Fluxo completo da aplicação

Quando o cliente envia:


```json
{
  "equipamento_id": "MOTOR-01",
  "temperatura": 88.5,
  "vibracao": 7.2,
  "horas_uso": 8100
}
```

acontece o seguinte:

```bash
1. FastAPI recebe o JSON
        ↓
2. LeituraEntrada valida tipos e limites
        ↓
3. LeituraSensor representa os dados do domínio
        ↓
4. ServicoPredicao coordena a operação
        ↓
5. ModeloFalha calcula a probabilidade
        ↓
6. ValidacaoMixin valida o resultado
        ↓
7. RepositorioSQLite salva no banco
        ↓
8. LogMixin registra a operação
        ↓
9. A API devolve a resposta
```

resposta aproximada:

```json
{
  "registro_id": 1,
  "equipamento_id": "MOTOR-01",
  "probabilidade_falha": 0.7635,
  "falha_prevista": true
}
```

In [16]:
for classe in ServicoPredicao.mro():
    print(classe.__name__)

ServicoPredicao
LogMixin
ValidacaoMixin
object
